In [ ]:
!pip install transformers datasets accelerate gdown

In [ ]:
#!gdown 1kJXtUko-70rNP250KxkiW3OTE7bg1RAo

In [ ]:
import numpy as np
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
df = pd.read_csv("final_dataset.csv")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.metrics import classification_report, f1_score

# ---------------------------------------------------------
# 1) Ucitavanje i priprema podataka
# ---------------------------------------------------------
df = pd.read_csv("final_dataset.csv")

# baciti redove bez teksta (nema sta trenirati)
df = df.dropna(subset=["SADRZAJ"]).reset_index(drop=True)

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]

def stance_label(row, topic):
    """samo stance: against / neutral / for (bez not_mentioned)"""
    stance = row[f"{topic}_stance"]
    if stance == 1:
        return "for"
    elif stance == -1:
        return "against"
    else:
        return "neutral"

# Za svaku temu pravimo poseban dataframe koji sadrzi SAMO redove gdje je
# tema zaista spomenuta (mentioned == 1/True). Redovi gdje tema NIJE
# spomenuta se u potpunosti izbacuju - BERT se ovdje trenira samo da
# prepozna stav (for/against/neutral), a ne da li je tema spomenuta.
df_topic_stance = {}
for topic in TOPICS:
    mask = df[f"{topic}_mentioned"].astype(bool)
    d = df.loc[mask].copy()
    d[f"{topic}_label"] = d.apply(lambda r: stance_label(r, topic), axis=1)
    df_topic_stance[topic] = d
    print(f"{topic}: {mask.sum()} od {len(df)} redova je 'mentioned' -> koristi se za stance klasifikaciju")

In [ ]:
for topic in TOPICS:
    d = df_topic_stance[topic]
    d["TEXT_BERT"] = (
        d["naslov"].fillna("") + ". " +
        d["SADRZAJ"].fillna("")
    )
    df_topic_stance[topic] = d

In [ ]:
from transformers import EarlyStoppingCallback

def train_bert_for_topic(topic, model_name="classla/bcms-bertic"):
    print("=" * 70)
    print(f"TEMA: {topic} | BERT FINE-TUNING (STANCE: against / neutral / for)")
    print("=" * 70)

    labels = ["against", "neutral", "for"]
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}

    temp = df_topic_stance[topic][["TEXT_BERT", f"{topic}_label"]].dropna().copy()
    temp["label"] = temp[f"{topic}_label"].map(label2id)

    train_df, test_df = train_test_split(
        temp,
        test_size=0.2,
        random_state=42,
        stratify=temp["label"]
    )

    train_ds = Dataset.from_pandas(train_df[["TEXT_BERT", "label"]])
    test_ds = Dataset.from_pandas(test_df[["TEXT_BERT", "label"]])

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch["TEXT_BERT"],
            truncation=True,
            padding="max_length",
            max_length=512
        )

    train_ds = train_ds.map(tokenize, batched=True)
    test_ds = test_ds.map(tokenize, batched=True)

    train_ds = train_ds.remove_columns(["TEXT_BERT", "__index_level_0__"])
    test_ds = test_ds.remove_columns(["TEXT_BERT", "__index_level_0__"])

    train_ds.set_format("torch")
    test_ds.set_format("torch")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id
    )

    def compute_metrics(eval_pred):
        logits, y_true = eval_pred
        y_pred = np.argmax(logits, axis=1)

        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        }

    args = TrainingArguments(
        output_dir=f"./bert_results_stance_{topic}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=10,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=1,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    pred = trainer.predict(test_ds)
    y_pred = np.argmax(pred.predictions, axis=1)
    y_true = pred.label_ids

    print(classification_report(
        y_true,
        y_pred,
        target_names=labels,
        zero_division=0
    ))

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    return {
        "trainer": trainer,
        "model": model,
        "tokenizer": tokenizer,
        "macro_f1": macro_f1,
        "label2id": label2id,
        "id2label": id2label,
    }

In [ ]:
bert_result_euro = train_bert_for_topic("euroatlantske_integracije")

In [ ]:
bert_result_gradanska = train_bert_for_topic("gradjanska_vs_konstitutivni")

In [ ]:
bert_result_izborna = train_bert_for_topic("izborna_reforma")

In [ ]:
bert_result_genocid = train_bert_for_topic("negiranje_genocida")

In [ ]:
results = {
    "euroatlantske_integracije": bert_result_euro,
    "gradjanska_vs_konstitutivni": bert_result_gradanska,
    "izborna_reforma": bert_result_izborna,
    "negiranje_genocida": bert_result_genocid,
}

for topic, res in results.items():
    save_path = f"./bertic_stance_{topic}_final"
    res["trainer"].save_model(save_path)
    res["tokenizer"].save_pretrained(save_path)
    print(f"Sacuvano: {save_path}")

In [ ]:
import os, shutil

os.makedirs("bertic_stance_models", exist_ok=True)

topics = [
    "euroatlantske_integracije",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
    "negiranje_genocida",
]

for topic in topics:
    src = f"./bertic_stance_{topic}_final"
    dst = f"./bertic_stance_models/{topic}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)

shutil.make_archive("bertic_stance_models", "zip", "bertic_stance_models")
print("Gotovo: bertic_stance_models.zip")

# Predict

In [ ]:
def predict(text, model, tokenizer, device):
    inputs = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()

    return model.config.id2label[pred_id], probs[0][pred_id].item()


def predict_stance(text, euro_m, izborna_m, genocid_m, grad_m,
                    euro_tok, izborna_tok, genocid_tok, grad_tok, device):

    euro_p = predict(text, euro_m, euro_tok, device)[0]
    izborna_p = predict(text, izborna_m, izborna_tok, device)[0]
    genocid_p = predict(text, genocid_m, genocid_tok, device)[0]
    grad_p = predict(text, grad_m, grad_tok, device)[0]

    return {
        "izborna_p": izborna_p,
        "euro_p": euro_p,
        "genocid_p": genocid_p,
        "grad_p": grad_p,
    }

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
euro_tok, euro_m = AutoTokenizer.from_pretrained("./bertic_stance_euroatlantske_integracije_final"), \
                   AutoModelForSequenceClassification.from_pretrained("./bertic_stance_euroatlantske_integracije_final").to(device).eval()

izborna_tok, izborna_m = AutoTokenizer.from_pretrained("./bertic_stance_izborna_reforma_final"), \
                         AutoModelForSequenceClassification.from_pretrained("./bertic_stance_izborna_reforma_final").to(device).eval()

genocid_tok, genocid_m = AutoTokenizer.from_pretrained("./bertic_stance_negiranje_genocida_final"), \
                         AutoModelForSequenceClassification.from_pretrained("./bertic_stance_negiranje_genocida_final").to(device).eval()

grad_tok, grad_m = AutoTokenizer.from_pretrained("./bertic_stance_gradjanska_vs_konstitutivni_final"), \
                   AutoModelForSequenceClassification.from_pretrained("./bertic_stance_gradjanska_vs_konstitutivni_final").to(device).eval()

**Napomena:** ovaj model uvijek vraća jedan od tri stava (`for`/`against`/`neutral`) -
ne zna prepoznati da tema uopšte nije spomenuta. Zato ga ima smisla koristiti
samo nad tekstovima za koje se već zna (npr. preko binary BERTIC modela) da je
tema zaista prisutna.

In [ ]:
text = """
Ovdje idi svoj tekst za testiranje.
"""

In [ ]:
result = predict_stance(text,
                         euro_m, izborna_m, genocid_m, grad_m,
                         euro_tok, izborna_tok, genocid_tok, grad_tok, device)
print(result)